# AI4Science Hippocampus

Run your Python on Snellius from your laptop -- no batch scripts, no SSH, no
environment setup. This notebook walks through what the platform does today:

| | |
|---|---|
| **1. Data staging** | Bring an OpenML or Hugging Face dataset onto Snellius once, for everyone |
| **2. Job orchestration** | Run a function on Snellius: with community packages (OpenML, Hugging Face), your own container, multi-node Ray, EESSI modules, or automatic cluster selection |
| **3. Agentic Community Archive** | Tools from the research community, reviewed and published for AI agents |

Every example runs as it is. Run the cells top to bottom.

## Setup

You need a Snellius account and a Slurm token. Put both in a `.env` file in
the root of the client repository (next to `.env.example`):

```bash
export AI4SCIENCE_BASE_URL="https://ai4science.dev.sdp.surf.nl"
export SLURM_USER="your_snellius_user"
export SLURM_TOKEN="..."
```

Get a token on a Snellius login node with `scontrol token lifespan=86400`
(valid for one day) and copy the part after `SLURM_JWT=`.
The token is your only credential: it submits jobs as you, and only you can
read your jobs' logs and results.

In [1]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # the .env of this repository

from ai4science_client import Ai4ScienceClient

client = Ai4ScienceClient()  # reads AI4SCIENCE_BASE_URL, SLURM_USER, SLURM_TOKEN
print(f"Connected to {client.base_url} as {client.user}")

Connected to https://ai4science.dev.sdp.surf.nl as juliusa


## 1. Data staging

Ask for a public dataset by name; the platform downloads it onto Snellius into
a shared, managed location, where every job can read it. One request, and the
whole team has it.

```text
  you (juliusa)          ai4science API           maintainer          Snellius (runs as wodsmgr)
   |  request dataset         |                        |                        |
   |------------------------->|  status: AWAITING REVIEW|                        |
   |                          |----------------------->|  approve or reject     |
   |                          |<-----------------------|                        |
   |                          |  staging job as wodsmgr ----------------------->|  download into
   |                          |                        |                        |  managed_datasets/<name>
   |                          |                        |                        |  read access (ACL): juliusa only
   |  status: READY           |<------------------------------------------------|
```

Staging is switched on per deployment by an administrator; if it is off, the
request is refused with a clear message.

In [ ]:
import time

import requests

DATASET_ID, PLATFORM, CONFIG = "glue", "Hugging Face", "mrpr"

resp = requests.post(
    f"{client.base_url}/datasets/add",
    json={
        "dataset_id": DATASET_ID,
        "platform": PLATFORM,
        "dataset_config_name": CONFIG,
        "user": client.user,
        "token": client.token,
    },
    timeout=60,
)
if resp.status_code == 403:
    print("Dataset staging is switched off on this deployment:", resp.json()["detail"])
else:
    resp.raise_for_status()
    print("Staging job submitted:", resp.json()["job_id"])

    name = DATASET_ID.rsplit("/", 1)[-1]
    for _ in range(90):  # up to 15 minutes
        status = requests.post(
            f"{client.base_url}/datasets/status", json={"datasets": [name]}, timeout=30
        ).json()[name]
        print(f"{time.strftime('%H:%M:%S')}  {status['status'].upper()}")
        if status["status"].upper() in ("READY", "FAILED"):
            print(status)
            break
        time.sleep(10)

Staging job submitted: 28921683
14:19:53  PENDING
14:20:04  PENDING
14:20:14  PENDING
14:20:24  PENDING
14:20:34  PENDING


## 2. Job orchestration

Write a normal Python function; the client runs it on Snellius and hands you
the return value. Imports go inside the function, and arguments and results
must be JSON-serialisable. `stream=True` shows the job's output live.

### Community packages: OpenML

`@job` turns a function into a Snellius job. Here an OpenML task is fetched,
a model trained and scored -- `dependencies` are installed for this job only.

In [2]:
from ai4science_client import job


@job(
    base_url=client.base_url,
    user=client.user,
    token=client.token,
    runtime="apptainer",
    dependencies=["openml", "scikit-learn"],
    stream=True,
)
def run_openml_task(task_id: int) -> dict:
    import openml
    from sklearn.ensemble import RandomForestClassifier

    task = openml.tasks.get_task(task_id)
    X, y = task.get_X_and_y()
    train, test = task.get_train_test_split_indices()
    model = RandomForestClassifier(n_estimators=100).fit(X[train], y[train])
    return {"task_id": task_id, "accuracy": model.score(X[test], y[test])}


print(run_openml_task(31))

=== [1/3] Preparing Apptainer Container ===
=== [2/3] Installing Dependencies (Ephemeral Overlay): openml scikit-learn ===
=== [3/3] Running Python Script: /projects/2/managed_datasets/containers/ai4science/jobs/api_job_a57aa247.py ===
/projects/2/managed_datasets/containers/ai4science/jobs/api_job_a57aa247.py:6: FutureWarning: Support for `dataset_format='array'` will be removed in 0.15,start using `dataset_format='dataframe' to ensure your code will continue to work. You can use the dataframe's `to_numpy` function to continue using numpy arrays.
  X, y = task.get_X_and_y()
/opt/conda/lib/python3.11/site-packages/openml/tasks/task.py:334: FutureWarning: Support for `dataset_format='array'` will be removed in 0.15,start using `dataset_format='dataframe' to ensure your code will continue to work. You can use the dataframe's `to_numpy` function to continue using numpy arrays.
  X, y, _, _ = dataset.get_data(
[result received: {'task_id': 31, 'accuracy': 0.8}]
INFO:    Cleanup error: whil

### Community models: Hugging Face, on a GPU, with your own file

`resources` asks for exactly what the job needs -- here one H100 GPU. A local
file passed as an `artifact` is uploaded for you and arrives in the function
as a normal path.

In [5]:
from ai4science_client.schemas import SlurmResourceConfig

with open("sample_texts.txt", "w") as f:
    f.write("Snellius makes large-scale AI research so much easier.\n")
    f.write("I really dislike waiting for slow pip installs.\n")
    f.write("This dataset looks promising for our next experiment.\n")


def classify(data_path: str) -> list:
    import torch
    from transformers import pipeline

    with open(data_path) as f:
        texts = [line.strip() for line in f if line.strip()]
    classifier = pipeline("sentiment-analysis", device=0 if torch.cuda.is_available() else -1)
    return [(text, r["label"], round(r["score"], 3)) for text, r in zip(texts, classifier(texts))]


result = client.run(
    classify,
    runtime="apptainer",
    artifacts={"data_path": "sample_texts.txt"},
    dependencies=["transformers<4.50"],
    resources=SlurmResourceConfig(
        partition="gpu_h100", tres_per_node="gres:gpu:1",
        cpus_per_task=8, memory_mb=32000, time_limit_minutes=20,
    ),
    stream=True,
)
for row in result:
    print(row)

=== [1/3] Preparing Apptainer Container ===
=== [2/3] Installing Dependencies (Ephemeral Overlay): transformers<4.50 ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 140.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 112.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.13.0
    Not uninstalling huggingface-hub at /opt/conda/lib/python3.11/site-packages, outside environment /scratch-local/.28914793/ai4science_venv
    Can't uninstall 'huggingface_hub'. No files were found to uninstall.
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Not uninstalling tokenizers at /opt/conda/lib/python3.11/site-packages, outside environment /scratch-local/.28914793/ai4science_venv
    Can't uninstall 'tokenizers'. No files were found to uninstall.
  Attempting uninstall: transformers
 

### Bring your own container

Run your function inside any image from SURF's registry (`cr.surf.nl`). The image is fetched
once and kept it in your home cache (`~/.cache/ai4science/containers`); the next
job starts right away.

The image below is a public one from another project. For a **private** image,user needs to pass auth token too.

In [6]:
def inside_the_image() -> dict:
    import platform

    import boto3  # comes with this image, not with ours

    return {"python": platform.python_version(), "boto3": boto3.__version__}


print(
    client.run(
        inside_the_image,
        container="cr.surf.nl/mcc-mcc/odissee-poc/preprocess:latest",
        resources=SlurmResourceConfig(
            partition="genoa", cpus_per_task=2, memory_mb=4000,
            time_limit_minutes=10, tres_per_node="",
        ),
        stream=True,
    )
)


# from ai4science_client.schemas import SlurmResourceConfig


# def inside_the_image() -> dict:
#     import platform

#     import fastapi  # comes with this image, not with the default one

#     return {"python": platform.python_version(), "fastapi": fastapi.__version__}


# print(
#     client.run(
#         inside_the_image,
#         container="cr.surf.nl/hpml-ai4science_poc/ai4science_poc:ffd53047",
#         resources=SlurmResourceConfig(
#             partition="genoa", cpus_per_task=2, memory_mb=4000,
#             time_limit_minutes=10, tres_per_node="",
#         ),
#         stream=True,
#     )
# )

=== [1/3] Preparing Apptainer Container ===
Container: cr.surf.nl/mcc-mcc/odissee-poc/preprocess:latest
Using cached image cr.surf.nl/mcc-mcc/odissee-poc/preprocess:latest
=== [2/3] Installing Dependencies (Ephemeral Overlay):  ===
=== [3/3] Running Python Script: /projects/2/managed_datasets/containers/ai4science/jobs/api_job_da675ea3.py ===
[result received: {'python': '3.12.16', 'boto3': '1.43.122'}]
=== Job Finished with exit code 0 ===
{'python': '3.12.16', 'boto3': '1.43.122'}


### Multi-node Ray

`runtime="ray"` starts a Ray cluster across the nodes you ask for before your
function runs; just connect with `ray.init(address="auto")`. `nodes` sets the
cluster size; the other resource fields apply to each node.

In [7]:
def ray_cluster_size() -> dict:
    import ray

    ray.init(address="auto")
    return {"nodes": len(ray.nodes()), "cpus": ray.cluster_resources()["CPU"]}


print(
    client.run(
        ray_cluster_size,
        runtime="ray",
        resources=SlurmResourceConfig(
            partition="rome", nodes=2, cpus_per_task=8, memory_mb=16000,
            time_limit_minutes=15, tres_per_node="",
        ),
        stream=True,
    )
)

=== [1/4] Preparing Ray Container ===
=== [2/4] Starting Ray head on tcn362 (172.18.57.116:6379) ===
=== [3/4] Starting 1 Ray worker(s) ===
Starting worker on tcn363
2026-11-05 00:18:07,028	INFO usage_lib.py:474 -- Usage stats collection is enabled by default without user confirmation because this terminal is detected to be non-interactive. To disable this, add `--disable-usage-stats` to the command that starts the cluster, or run the following command: `ray disable-usage-stats` before starting the cluster. See https://docs.ray.io/en/master/cluster/usage-stats.html for more details.
2026-11-05 00:18:07,030	INFO scripts.py:1042 -- Local node IP: 172.18.57.116
2026-11-05 00:18:10,905	SUCC scripts.py:1081 -- --------------------
2026-11-05 00:18:10,906	SUCC scripts.py:1082 -- Ray runtime started.
2026-11-05 00:18:10,906	SUCC scripts.py:1083 -- --------------------
2026-11-05 00:18:10,906	INFO scripts.py:1085 -- Next steps
2026-11-05 00:18:10,906	INFO cli_output_helpers.py:16 -- Note: The 

### EESSI software modules

No container, no installs: `runtime="eessi"` loads ready-built, optimised
software from EESSI. Name the modules; your function runs with them.

In [8]:
def matrix_sum() -> float:
    import numpy as np

    return float(np.random.rand(1000, 1000).sum())


print(
    client.run(
        matrix_sum,
        runtime="eessi",
        modules=["foss/2023a", "SciPy-bundle/2023.07-gfbf-2023a"],
        stream=True,
    )
)

--- Phase 1: EESSI Init ---
Attempting to initialize EESSI version: 2023.06
EESSI Initialized successfully.
--- Phase 2: Module Load ---
Loading foss/2023a...
Loading SciPy-bundle/2023.07-gfbf-2023a...
--- Phase 3: Python ---
Using Python from: /cvmfs/software.eessi.io/versions/2023.06/software/linux/x86_64/amd/zen4/software/Python/3.11.3-GCCcore-12.3.0/bin/python3
[result received: 499733.66375561274]
=== Job Finished with exit code 0 ===
499733.66375561274


### Automatic tier routing

Not sure what to ask for? With `tier="auto"` the platform estimates what the
job needs (CPU, memory, GPU) from its code and dependencies, and picks the
smallest cluster and partition that fit. This job uses PyTorch on a GPU, so it
lands on a Snellius GPU partition.

In [9]:
def gpu_matmul(size: int) -> dict:
    import time

    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    a = torch.randn(size, size, device=device)
    start = time.time()
    (a @ a).sum().item()
    return {"device": device, "seconds": round(time.time() - start, 3)}


print(client.run(gpu_matmul, 2048, dependencies=["torch"], tier="auto", stream=True))

=== [1/3] Preparing Apptainer Container ===
=== [2/3] Installing Dependencies (Ephemeral Overlay): torch ===
=== [3/3] Running Python Script: /projects/2/managed_datasets/containers/ai4science/jobs/api_job_9a4261a7.py ===
[result received: {'device': 'cuda', 'seconds': 0.263}]
=== Job Finished with exit code 0 ===
{'device': 'cuda', 'seconds': 0.263}


### Submit now, check later -- with MLflow tracking

`client.run` waits for the result. `client.submit` returns at once with a job
you can check on later, also from another session with `client.job(job_id)`.
`track=True` records the run in MLflow.

In [12]:
from ai4science_client import build_script


def where_am_i() -> dict:
    import os

    return {"node": os.uname().nodename, "partition": os.environ.get("SLURM_JOB_PARTITION")}


submitted = client.submit(
    build_script(where_am_i),
    runtime="apptainer",
    resources=SlurmResourceConfig(
        partition="genoa", cpus_per_task=2, memory_mb=4000,
        time_limit_minutes=10, tres_per_node="",
    ),
    track=True,
    experiment_name="demo-notebook",
)
print("Submitted:", submitted.job_id)

Submitted: 08df4545aaef457cb42b30fe91c90657


In [14]:
later = client.job(submitted.job_id)  # works from any session
result = later.wait(timeout=1800)
print("Result:", result.status, result.result)
# print("Log:\n" + (later.logs() or ""))

run = later.tracking_run()
print("MLflow run:", run["run_id"], run["status"], run["params"])

Result: completed {'node': 'tcn546.local.snellius.surf.nl', 'partition': 'genoa'}
MLflow run: 4110d1b2ae8c4ec5bafb6c45f1aa6f44 FINISHED {'dependencies': '', 'resources.partition': 'genoa', 'resources.cpus_per_task': '2', 'resources.memory_mb': '4000', 'resources.time_limit_minutes': '10', 'resources.tres_per_node': ''}


## 3. Agentic Community Archive

Researchers share tools with each other -- and with AI agents. A contributor
submits a tool (a manifest plus its code); a maintainer reviews it; once
approved, it is published and every agent connected to the platform can find
it.

```text
  contributor                  maintainer                    AI agents (MCP)
      |  submit tool               |                                |
      |--------------------------->|  review: approve or reject     |
      |     status: submitted      |------------------------------->|  published:
      |                            |                                |  discoverable in
      |                            |                                |  the archive
```

Agents connect over MCP at `{base_url}/mcp`. `Ai4ScienceMCPClient` shows you
exactly what they see.

In [15]:
from ai4science_client import Ai4ScienceMCPClient

async with Ai4ScienceMCPClient(base_url=client.base_url) as mcp:
    tools = await mcp.list_tools()
    archive = await mcp.list_community_tools()

print("What an agent can do:")
for tool in tools:
    print(f"  {tool.name:26} {tool.title}")

print("\nPublished community tools:")
for tool in archive:
    print(f"  {tool.qualified_name:32} v{tool.version}  {tool.title}")

What an agent can do:
  get_server_specs           Get server info
  list_compute_targets       List compute targets
  estimate_job_resources     Estimate job resources
  select_compute_target      Select compute target
  list_community_tools       List community tools

Published community tools:
  chemistry.rdkit_descriptors      v1.0.0  RDKit molecular descriptors
  demo.word_counter                v1.0.1791367061  Word counter


### Contribute a tool

A tool is a manifest -- what it does, its inputs, how it runs -- plus its
source files. Versions are immutable, so every run of this cell submits a new
one.

In [17]:
import time

import requests

API = f"{client.base_url}/api/v1/community-tools"
version = f"1.0.{int(time.time())}"

submission = {
    "submitted_by": client.user,
    "manifest": {
        "domain": "demo", "name": "word_counter", "version": version,
        "title": "Word counter", "description": "Counts the words in a piece of text.",
        "authors": [client.user], "license": "MIT",
        "input_schema": {"type": "object", "properties": {"text": {"type": "string"}}},
        "runtime": {"kind": "ephemeral", "entrypoint": "main.py"},
    },
    "files": {
        "main.py": (
            "import json\n"
            "text = json.load(open('input.json'))['text']\n"
            "json.dump({'words': len(text.split())}, open('output.json', 'w'))\n"
        )
    },
}
record = requests.post(f"{API}/submissions", json=submission, timeout=30)
record.raise_for_status()
print(f"demo.word_counter v{version}:", record.json()["status"])

demo.word_counter v1.0.1793867611: submitted


### Review and publish (maintainers)

Maintainers approve with their reviewer key (`AI4SCIENCE_REVIEWER_KEY` in
`.env`). Without one, this cell only shows the submission's status: it stays
`submitted` until a maintainer has looked at it.

In [18]:
import os

import requests

from ai4science_client import Ai4ScienceMCPClient

key = os.environ.get("AI4SCIENCE_REVIEWER_KEY")
path = f"{API}/submissions/demo/word_counter/{version}"

if key:
    review = requests.post(
        f"{path}/review",
        json={"decision": "approve", "reviewed_by": client.user},
        headers={"X-Reviewer-Key": key},
        timeout=30,
    )
    review.raise_for_status()
    print("Reviewed:", review.json()["status"])
else:
    print("No reviewer key; status:", requests.get(path, timeout=30).json()["status"])

async with Ai4ScienceMCPClient(base_url=client.base_url) as mcp:
    found = await mcp.list_community_tools(query="word")
print("Agents now find:", [f"{t.qualified_name} v{t.version}" for t in found])

Reviewed: published
Agents now find: ['demo.word_counter v1.0.1793867611']
